# Ekşi Sözlük: The Master Analytics Engine 🚀
**Date:** May 14, 2026 | **Dataset:** 12.9M entries · 202K authors · 28K topics · 1999–2026

### Dashboards
1. **The Pulse** — Historical volume
2. **The Trend Engine** — Topics exploding this week (relative to dataset max)
3. **The First Mover Advantage** — Does position 1 always win?
4. **Effort vs Reward** — Word count density heatmap
5. **The Provocation Effect** — Does `?` drive engagement?
6. **Author Clout** — Do legends win by default?
7. **The Night Owl Effect** — When does the platform peak?
8. **Day-of-Week Rhythm** — Which day produces the most viral entries?
9. **Topic Gravity** — Which topics generate the deepest discussions?
10. **Favorites Inequality** — Gini coefficient: how unfair is the attention economy?

In [ ]:
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
ENTRIES_CSV = os.path.join(BASE_DIR, 'eksifikir_latest_entries.csv')

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")

con.execute(f"""
CREATE OR REPLACE VIEW cleaned_entries AS
SELECT
    *,
    regexp_replace(topic_title, '(\\S)\\d+$', '\\1') AS clean_topic_title,
    TRY_CAST(created_at AS TIMESTAMP) AS safe_created_at
FROM read_csv_auto('{ENTRIES_CSV}', ignore_errors=True)
WHERE TRY_CAST(created_at AS TIMESTAMP) IS NOT NULL
""")

print('✅ Master Engine Initialized. 12.9M rows ready.')

## 📈 1. The Pulse (Historical Volume)
Is the platform growing or dying?

In [ ]:
df_pulse = con.execute("""
SELECT date_trunc('month', safe_created_at) AS month, count(*) AS entry_count
FROM cleaned_entries WHERE safe_created_at >= '2010-01-01'
GROUP BY 1 ORDER BY 1
""").pl()

fig = px.area(df_pulse.to_pandas(), x='month', y='entry_count',
              title='Ekşi Sözlük: Monthly Entry Volume (2010–2026)',
              color_discrete_sequence=['#2ecc71'],
              labels={'month': 'Month', 'entry_count': 'Entries'})
fig.update_layout(template='plotly_dark', hovermode='x unified')
fig.show()

## 🔥 2. The Trend Engine (Velocity)
Topics exploding this week vs their all-time average.

In [ ]:
df_trends = con.execute("""
WITH max_date AS (SELECT max(safe_created_at) AS t FROM cleaned_entries),
historical AS (
    SELECT clean_topic_title AS title, count(*)/1000.0 AS hist_avg
    FROM cleaned_entries, max_date
    WHERE safe_created_at < max_date.t - INTERVAL '7 days' GROUP BY 1
),
recent AS (
    SELECT clean_topic_title AS title, count(*)/7.0 AS recent_avg
    FROM cleaned_entries, max_date
    WHERE safe_created_at >= max_date.t - INTERVAL '7 days' GROUP BY 1
)
SELECT r.title,
       round(r.recent_avg, 2) AS daily_entries_last_7d,
       round(r.recent_avg / NULLIF(h.hist_avg, 0), 1) AS velocity_multiplier
FROM recent r JOIN historical h ON r.title = h.title
WHERE h.hist_avg > 0.05 AND r.recent_avg > 5
ORDER BY velocity_multiplier DESC LIMIT 15
""").pl()

if len(df_trends) == 0:
    print('No trending topics found in the last 7 days of the dataset.')
else:
    fig = px.bar(df_trends.to_pandas(), x='velocity_multiplier', y='title', orientation='h',
                 title='Trending Topics: Velocity Multiplier (Last 7 Days vs History)',
                 color='velocity_multiplier', color_continuous_scale='Inferno',
                 labels={'velocity_multiplier': 'Velocity (x)', 'title': 'Topic'})
    fig.update_layout(template='plotly_dark', yaxis={'categoryorder': 'total ascending'})
    fig.show()

## 🥇 3. The First Mover Advantage
Does being the 1st entry in a topic guarantee more favorites?

In [ ]:
df_pos = con.execute("""
WITH ranked AS (
    SELECT favorites,
           row_number() OVER (PARTITION BY topic_id ORDER BY safe_created_at ASC) AS pos
    FROM cleaned_entries
)
SELECT pos AS position_in_topic, round(avg(favorites), 2) AS avg_favs
FROM ranked WHERE pos <= 50
GROUP BY 1 ORDER BY 1
""").pl()

fig = px.line(df_pos.to_pandas(), x='position_in_topic', y='avg_favs', markers=True,
              title='The First Mover Advantage: Avg Favorites by Entry Position',
              labels={'position_in_topic': 'Position in Topic', 'avg_favs': 'Avg Favorites'})
fig.update_layout(template='plotly_dark')
fig.show()

## ✍️ 4. Effort vs Reward (Word Count Density)
Do long essays beat witty one-liners?

In [ ]:
df_effort = con.execute("""
SELECT favorites,
       length(text) - length(replace(text, ' ', '')) + 1 AS approx_word_count
FROM cleaned_entries
WHERE favorites > 20 AND text IS NOT NULL
USING SAMPLE 100000 ROWS
""").pl()

fig = px.density_heatmap(df_effort.to_pandas(), x='approx_word_count', y='favorites',
                         nbinsx=40, nbinsy=40,
                         title='Effort vs Reward: Word Count vs Favorites (High-Performers)',
                         labels={'approx_word_count': 'Word Count', 'favorites': 'Favorites'},
                         color_continuous_scale='Inferno')
fig.update_layout(template='plotly_dark', xaxis_range=[0, 800], yaxis_range=[20, 500])
fig.show()

## 🗣️ 5. The Provocation Effect
Does asking a question (?) drive more favorites?

In [ ]:
df_q = con.execute("""
SELECT
    CASE WHEN text LIKE '%?%' THEN 'Has Question Mark (?)'
         ELSE 'No Question Mark' END AS text_type,
    round(avg(favorites), 2) AS avg_favs,
    count(*) AS total_entries
FROM cleaned_entries GROUP BY 1
""").pl()

fig = px.bar(df_q.to_pandas(), x='text_type', y='avg_favs', color='text_type', text='avg_favs',
             title='The Provocation Effect: Does Asking Questions Drive Engagement?',
             color_discrete_sequence=['#e74c3c', '#3498db'])
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

## 👑 6. Author Clout (Status vs Quality)
Do Legendary authors win automatically?

In [ ]:
df_clout = con.execute("""
WITH entry_data AS (
    SELECT favorites, count(*) OVER (PARTITION BY author) AS total_posts
    FROM cleaned_entries
)
SELECT
    CASE
        WHEN total_posts >= 5000 THEN '1. Legend (5000+ posts)'
        WHEN total_posts >= 1000 THEN '2. Elite (1000+ posts)'
        WHEN total_posts >= 100  THEN '3. Regular (100+ posts)'
        ELSE '4. Casual (<100 posts)'
    END AS author_tier,
    round(avg(favorites), 2) AS avg_favs,
    count(*) AS total_entries
FROM entry_data GROUP BY 1 ORDER BY 1
""").pl()

fig = px.bar(df_clout.to_pandas(), x='author_tier', y='avg_favs', color='author_tier', text='avg_favs',
             title='Author Clout: Average Favorites by Experience Tier',
             color_discrete_sequence=['#f1c40f', '#e67e22', '#3498db', '#95a5a6'])
fig.update_layout(template='plotly_dark', showlegend=False)
fig.show()

## 🌙 7. The Night Owl Effect (Hourly Pattern)
When does the platform peak? When do viral entries get written?

In [ ]:
df_hourly = con.execute("""
SELECT
    hour(safe_created_at) AS hour_of_day,
    count(*) AS entry_count,
    round(avg(favorites), 3) AS avg_favs
FROM cleaned_entries
GROUP BY 1 ORDER BY 1
""").pl()

pdf = df_hourly.to_pandas()
fig = go.Figure()
fig.add_trace(go.Bar(x=pdf['hour_of_day'], y=pdf['entry_count'],
                     name='Entry Volume', marker_color='rgba(52,152,219,0.7)', yaxis='y1'))
fig.add_trace(go.Scatter(x=pdf['hour_of_day'], y=pdf['avg_favs'],
                         name='Avg Favorites', mode='lines+markers',
                         marker=dict(color='#f39c12', size=8), yaxis='y2'))
fig.update_layout(
    template='plotly_dark',
    title='The Night Owl Effect: Hourly Activity & Engagement (UTC+3)',
    xaxis=dict(title='Hour of Day (UTC+3)', tickmode='linear', dtick=1),
    yaxis=dict(title='Entry Count'),
    yaxis2=dict(title='Avg Favorites', overlaying='y', side='right', showgrid=False),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

## 📅 8. Day-of-Week Rhythm
Which day produces the most entries AND the most viral content?

In [ ]:
df_dow = con.execute("""
SELECT
    dayofweek(safe_created_at) AS dow_num,
    CASE dayofweek(safe_created_at)
        WHEN 0 THEN 'Sun' WHEN 1 THEN 'Mon' WHEN 2 THEN 'Tue'
        WHEN 3 THEN 'Wed' WHEN 4 THEN 'Thu' WHEN 5 THEN 'Fri' ELSE 'Sat'
    END AS day_name,
    count(*) AS entry_count,
    round(avg(favorites), 3) AS avg_favs,
    count(*) FILTER (WHERE favorites >= 100) AS viral_entries
FROM cleaned_entries
GROUP BY 1, 2 ORDER BY 1
""").pl()

pdf = df_dow.to_pandas()
fig = px.bar(pdf, x='day_name', y='entry_count', color='avg_favs',
             title='Day-of-Week Rhythm: Volume vs Engagement Quality',
             color_continuous_scale='Plasma',
             labels={'day_name': 'Day', 'entry_count': 'Entry Count', 'avg_favs': 'Avg Favs'},
             text='viral_entries')
fig.update_traces(texttemplate='%{text} viral', textposition='outside')
fig.update_layout(template='plotly_dark')
fig.show()

## 🌌 9. Topic Gravity (Depth of Discussion)
Top topics by total entries AND average favorites — which have both volume AND quality?

In [ ]:
df_gravity = con.execute("""
SELECT
    clean_topic_title AS topic,
    count(*) AS total_entries,
    round(avg(favorites), 2) AS avg_favs,
    max(favorites) AS peak_favs,
    count(*) FILTER (WHERE favorites >= 50) AS hot_entries
FROM cleaned_entries
GROUP BY 1
HAVING count(*) >= 500
ORDER BY hot_entries DESC
LIMIT 25
""").pl()

fig = px.scatter(df_gravity.to_pandas(), x='total_entries', y='avg_favs',
                 size='hot_entries', color='peak_favs', hover_name='topic',
                 title='Topic Gravity: Volume vs Quality (bubble = hot entries ≥50 favs)',
                 color_continuous_scale='Viridis',
                 labels={'total_entries': 'Total Entries', 'avg_favs': 'Avg Favorites', 'peak_favs': 'Peak Favorites'})
fig.update_layout(template='plotly_dark')
fig.show()

## ⚖️ 10. Favorites Inequality (Gini Coefficient)
How unfair is the Ekşi attention economy? Computed per year.

In [ ]:
df_gini_raw = con.execute("""
SELECT year(safe_created_at) AS yr, favorites
FROM cleaned_entries
WHERE year(safe_created_at) >= 2010
USING SAMPLE 500000 ROWS
""").pl()

def gini(arr):
    a = sorted([x for x in arr if x >= 0])
    n = len(a)
    if n == 0 or sum(a) == 0:
        return 0
    cumsum = sum(v * (2*(i+1) - n - 1) for i, v in enumerate(a))
    return cumsum / (n * sum(a))

results = []
for yr, grp in df_gini_raw.group_by('yr'):
    results.append({'year': yr[0], 'gini': round(gini(grp['favorites'].to_list()), 4)})

df_gini = pd.DataFrame(sorted(results, key=lambda x: x['year']))

fig = px.line(df_gini, x='year', y='gini', markers=True,
              title='Favorites Inequality: Gini Coefficient by Year',
              labels={'year': 'Year', 'gini': 'Gini Coefficient'},
              color_discrete_sequence=['#e74c3c'])
fig.update_layout(template='plotly_dark', yaxis_range=[0, 1])
fig.add_hline(y=0.5, line_dash='dash', line_color='gray', annotation_text='0.5 reference')
fig.show()